<a href="https://colab.research.google.com/github/Murcha1990/GenModels_AI26/blob/main/Seminars_Base/Sem5_StDiffusion_AR/TextDiffusion_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Текстовая диффузия

Модель: [`dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1`](https://huggingface.co/dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1), ~0.6B параметров.

**Откуда она взялась.** Это обычная авторегрессионная Qwen3-0.6B, которую **переучили в маскированную диффузионную модель** (как LLaDA): дообучили на диалогах восстанавливать замаскированные токены по всему контексту, а не предсказывать следующий. Сделано в открытом фреймворке [dLLM](https://github.com/ZHZisZZ/dllm). Алгоритм генерации тот же, что у LLaDA из доклада, просто модель в ~13 раз меньше — поэтому она помещается в бесплатную T4, но и отвечает заметно слабее.

**Напоминание, как работает маскированная диффузия:**
* «шум» — это токен `[MASK]`;
* генерация начинается с ответа, целиком состоящего из масок;
* на каждом шаге модель (трансформер **без** каузальной маски) предсказывает **все** маски сразу, мы открываем только самые уверенные, а остальные оставляем масками;
* за `steps` шагов все маски открыты.

**Что посмотрим:**
1. Как ответ «проявляется» шаг за шагом.
2. В каком порядке открываются токены (не слева направо!).
3. Как число шагов влияет на скорость и качество.
4. Заполнение пропуска в середине текста — задача, естественная для диффузии.

In [1]:
# Версия transformers, с которой работает код модели (её же использует фреймворк dLLM).
# После установки Colab может попросить перезапустить сессию: Runtime → Restart session,
# затем запускайте ноутбук со следующей ячейки.
!pip install -q transformers==4.57.0 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.2 MB/s eta 0:00:00
Reason for being yanked: Error in the setup causing installation issues
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import time, html
import torch
import matplotlib.pyplot as plt
from matplotlib import cm
from IPython.display import HTML, display
import sys, types
import transformers
from transformers import AutoModelForMaskedLM, AutoTokenizer

# В файле модели есть строка `import dllm`, но только в тестовом блоке `if __name__ == "__main__"`,
# при загрузке она не выполняется. transformers же заранее проверяет все import в файле и падает,
# а пакета dllm нет на PyPI. Подставляем пустой модуль-заглушку — модели он не нужен.
sys.modules.setdefault("dllm", types.ModuleType("dllm"))
print("transformers", transformers.__version__)

MODEL = "dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# T4 не умеет bf16 аппаратно, а модель маленькая — берём float32 (~3 ГБ), так надёжнее всего
model = AutoModelForMaskedLM.from_pretrained(MODEL, trust_remote_code=True,
                                             dtype=torch.float32).to(device).eval()

MASK_ID = tokenizer.mask_token_id   # id токена [MASK]
print("Модель загружена, MASK_ID =", MASK_ID, "| GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "нет")

transformers 4.57.0


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

modeling_qwen3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/dllm-hub/Qwen3-0.6B-diffusion-mdlm-v0.1:
- modeling_qwen3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Модель загружена, MASK_ID = 151669 | GPU: Tesla T4


## Сэмплер

Пишем свой короткий сэмплер (по мотивам официального сэмплера LLaDA и примера из карточки модели), чтобы было видно каждую деталь.

Функция `denoise` получает последовательность, где неизвестные места заполнены `[MASK]`, и за `steps` шагов открывает все маски:
1. прогоняем всю последовательность через модель → для каждой позиции распределение по словарю;
2. для каждой маски берём самый вероятный токен и его вероятность («уверенность»);
3. открываем `k` самых уверенных масок, остальные остаются масками (**low-confidence remasking**).

Каждый шаг — **один** проход модели, а открывается сразу несколько токенов.

In [3]:
@torch.no_grad()
def denoise(x, steps, region=None, history=None):
    '''x: тензор [1, L] с MASK_ID на неизвестных местах.
    region=(lo, hi): открывать маски только внутри этого отрезка.
    history: список, куда складываем состояние после каждого шага.'''
    lo, hi = region if region is not None else (0, x.shape[1])
    n_masks = int((x[0, lo:hi] == MASK_ID).sum())
    # сколько токенов открывать на каждом шаге: поровну
    base, extra = divmod(n_masks, steps)
    schedule = [base + (1 if i < extra else 0) for i in range(steps)]

    for k in schedule:
        if k == 0:
            continue
        logits = model(x).logits[0].float()                 # [L, V] — предсказания для ВСЕХ позиций
        pred = logits.argmax(-1)                             # самый вероятный токен на каждой позиции
        conf = torch.softmax(logits, -1).gather(-1, pred[:, None]).squeeze(-1)  # его вероятность

        can_open = (x[0] == MASK_ID)                         # открывать можно только маски...
        can_open[:lo] = False; can_open[hi:] = False         # ...и только внутри отрезка
        conf[~can_open] = -float("inf")

        idx = conf.topk(k).indices                           # k самых уверенных масок
        x[0, idx] = pred[idx]
        if history is not None:
            history.append(x[0].clone().cpu())
    return x


def generate(question, gen_length=64, steps=64, block_length=32):
    '''Чат-генерация как в LLaDA: ответ длины gen_length из масок,
    открываем блоками по block_length слева направо (внутри блока — диффузия).'''
    msgs = [{"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user", "content": question}]
    # enable_thinking=False — так рекомендует карточка модели
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False,
                                           enable_thinking=False)
    prompt_ids = torch.tensor(tokenizer(prompt, add_special_tokens=False)["input_ids"], device=device)[None]
    P = prompt_ids.shape[1]

    x = torch.full((1, P + gen_length), MASK_ID, dtype=torch.long, device=device)
    x[:, :P] = prompt_ids

    n_blocks = gen_length // block_length
    history = [x[0].clone().cpu()]
    for b in range(n_blocks):
        lo, hi = P + b * block_length, P + (b + 1) * block_length
        denoise(x, steps // n_blocks, region=(lo, hi), history=history)
    return x, P, history

### Вспомогательные функции для отображения

`▒` — это ещё не открытая маска. Служебные токены (конец ответа) не печатаем.

In [4]:
def render(ids, start):
    out = []
    for t in ids[start:].tolist():
        out.append("▒" if t == MASK_ID else tokenizer.decode([t], skip_special_tokens=True))
    return "".join(out)


def show_order(history, start):
    '''Раскрашивает токены финального ответа по шагу, на котором они открылись:
    светлые — открыты рано, тёмные — поздно.'''
    final = history[-1]
    opened_at = []
    for pos in range(start, len(final)):
        step = next(i for i, h in enumerate(history) if h[pos] != MASK_ID)
        opened_at.append(step)
    n = max(opened_at)
    spans = []
    for pos, s in zip(range(start, len(final)), opened_at):
        tok = tokenizer.decode([final[pos]], skip_special_tokens=True)
        if not tok:
            continue
        r, g, b, _ = cm.Blues(0.15 + 0.8 * s / n)
        color = "white" if s / n > 0.55 else "black"
        spans.append(f'<span title="шаг {s}" style="background:rgb({r*255:.0f},{g*255:.0f},{b*255:.0f});'
                     f'color:{color};padding:2px 1px;border-radius:3px">{html.escape(tok)}</span>')
    display(HTML('<div style="font-family:monospace;line-height:2;font-size:15px">' + "".join(spans) + "</div>"))

## 1. Ответ «проявляется» из масок

In [8]:
question = "Explain in two or three sentences why the sky is blue."

x, P, history = generate(question, gen_length=64, steps=64, block_length=32)

for i in [0, 2, 4, 8, 12, 16, 24, 28, 32, 40, 56, 64]:
    print(f"шаг {i:>2}: {render(history[i], P)}")

шаг  0: ▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг  2: The▒▒ blue▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг  4: The sky appears blue▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг  8: The sky appears blue because of the way▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг 12: The sky appears blue because of the way sunlight▒ scattered▒ the atmosphere▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг 16: The sky appears blue because of the way sunlight is scattered by the atmosphere. When▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг 24: The sky appears blue because of the way sunlight is scattered by the atmosphere. When sunlight enters the atmosphere, it travels in▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
шаг 28: The sky appears blue because of the way sunlight is scattered by the atmosphere. When sunlight enters the atmosphere, it travels in a smaller angle than▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
ша

**Что видно:** ответ длины 64 изначально весь из масок. Первые 32 позиции (первый блок) заполняются за шаги 1–32, вторые — за шаги 33–64. Внутри блока токены появляются **не по порядку**.

## 2. В каком порядке открывались токены

Светлые токены открыты рано, тёмные — поздно (как на слайде из статьи).

In [7]:
show_order(history, P)

Обычно сначала появляются «опорные» токены и очевидные слова и знаки препинания, а содержательные детали — позже.

Попробуйте ту же визуализацию на задаче со слайда из статьи LLaDA (маленькая модель может ошибиться в арифметике — это тоже повод для обсуждения):

In [9]:
lily = ("Lily can run 12 kilometers per hour for 4 hours. After that, she runs 6 kilometers per hour. "
        "How many kilometers can she run in 8 hours?")
x, P, history_lily = generate(lily, gen_length=64, steps=64, block_length=32)
show_order(history_lily, P)

## 3. Меньше шагов → быстрее, но хуже?

При `steps = gen_length` модель открывает по 1 токену за шаг. Если шагов в 4 раза меньше, за шаг открывается по 4 токена — проходов модели в 4 раза меньше.

In [10]:
for steps in [64, 32, 16, 8]:
    torch.cuda.synchronize(); t0 = time.time()
    x, P, _ = generate(question, gen_length=64, steps=steps, block_length=32)
    torch.cuda.synchronize(); dt = time.time() - t0
    print(f"steps={steps:>2} | токенов за шаг: {64 // steps} | {dt:5.2f} с")
    print("   ", render(x[0], P).strip().replace("\n", " "))
    print()

steps=64 | токенов за шаг: 1 |  3.72 с
    The sky appears blue because of the way sunlight is scattered by the atmosphere. When sunlight enters the atmosphere, it travels in a smaller angle than it should, causing it to scatter more sunlight.

steps=32 | токенов за шаг: 2 |  1.86 с
    The sky appears blue because to our eyes, sunlight travels scattered more in atmosphere than in directions. This is because the atmosphere is a layer of gases in that scatter sunlight, making it appear as blue.

steps=16 | токенов за шаг: 4 |  0.92 с
    The sky appears blue because of atmosphere's scattering scattering of sunlightatters in the atmosphere, causing it shorter wavelengths blue than than light. This blue light is the the reason the sky appears blue blue.

steps= 8 | токенов за шаг: 8 |  0.46 с
    The sky appears blue because the the's light are scattered scattered atmosphere scattered atmosphere atmosphere atmosphere atmosphere by atmosphere atmosphere atmosphere atmosphere atmosphere atm

**Что обычно видно:** время падает почти пропорционально числу шагов, а при слишком малом числе шагов качество ухудшается — появляются повторы и ошибки. Причина: токены, открытые на одном шаге, предсказываются **независимо** друг от друга, и они могут плохо согласоваться между собой.

Заметьте ещё: у диффузионной модели нет KV-кэша, на каждом шаге вся последовательность прогоняется заново. Поэтому один шаг дороже, чем один шаг авторегрессионной модели.

## 4. Заполнение пропуска в середине

Модель видит текст **с обеих сторон**, поэтому может заполнить дыру в середине так, чтобы она согласовалась и с началом, и с концом. Для GPT, которая пишет слева направо, это неудобная задача.

Здесь без чат-шаблона: просто текст, в середине которого маски.

In [11]:
def infill(prefix, suffix, n_masks=8, steps=8):
    pre = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    suf = tokenizer(suffix, add_special_tokens=False)["input_ids"]
    bos = [tokenizer.bos_token_id] if tokenizer.bos_token_id is not None else []
    pre = bos + pre
    ids = pre + [MASK_ID] * n_masks + suf
    x = torch.tensor(ids, device=device)[None]
    history = [x[0].clone().cpu()]
    denoise(x, steps, region=(len(pre), len(pre) + n_masks), history=history)
    print("было:  ", render(history[0], len(bos)))
    print("стало: ", render(history[-1], len(bos)))
    print()

infill("The capital of France is", ", which is famous for the Eiffel Tower.", n_masks=3, steps=3)
infill("Yesterday I went to the", "and bought three apples and a loaf of bread.", n_masks=4, steps=4)
infill("def add(a, b):\n    return", "\n\nprint(add(2, 3))  # 5", n_masks=4, steps=4)

было:   The capital of France is▒▒▒, which is famous for the Eiffel Tower.
стало:  The capital of France is Paris, France, which is famous for the Eiffel Tower.

было:   Yesterday I went to the▒▒▒▒and bought three apples and a loaf of bread.
стало:  Yesterday I went to the store and,,and bought three apples and a loaf of bread.

было:   def add(a, b):
    return▒▒▒▒

print(add(2, 3))  # 5
стало:  def add(a, b):
    return a, and b

print(add(2, 3))  # 5



Число масок нужно задать заранее: длину пропуска модель сама не выбирает (это одно из ограничений из раздела «What is the catch?»). Если масок больше, чем нужно, модель обычно заполняет лишнее служебными токенами или пробелами.

## Попробуйте сами

* Задайте свой вопрос в `generate`. Базовая Qwen3 знает русский, но дообучение было в основном на английских данных — попробуйте и сравните.
* Увеличьте `gen_length` до 128 (и `steps` до 128), чтобы получить ответ подлиннее.
* Поменяйте `block_length`: при `block_length = gen_length` вся генерация — один блок, порядок открытия токенов становится ещё свободнее.
* Поставьте `steps=4` и посмотрите, как разваливается ответ.